# 👥 NB3 — WorldPop Population API
**Source:** `https://api.worldpop.org/v1/services/stats`  
**Protocol:** HTTP GET  
**Schedule:** Once per event (population is static)  
**Auth:** None

WorldPop is a research programme based in the School of Geography and Environmental Science, University of Southampton.[1] The programme employs a multidisciplinary team of researchers, analysts, GIS technicians, and project specialists who construct open data on populations and population attributes at high spatial resolution.
https://en.wikipedia.org/wiki/WorldPop_Project 

Fetches the total population within a bounding box around the epicentre (2020 dataset). Used to estimate how many people likely felt the earthquake.

In [2]:
!pip install requests pandas matplotlib --quiet


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import requests, json, math, time, urllib.parse
import pandas as pd
import matplotlib.pyplot as plt
import datetime

UTC = datetime.timezone.utc
def utcnow(): return datetime.datetime.now(UTC)

HEADERS = {"User-Agent": "EQMonitor/1.0"}

# Epicentre from EMSC event (Fiji M5.2)
LAT, LON = -17.7134, 178.0650
MAG, DEPTH = 5.2, 226.8
DEG = 2.7  # ~300km bounding box half-width

print("✅ Config ready")

✅ Config ready


## Build Request

WorldPop accepts a GeoJSON polygon. We pass a bounding box around the epicentre.

In [4]:
coords = [
    [LON-DEG, LAT-DEG], [LON+DEG, LAT-DEG],
    [LON+DEG, LAT+DEG], [LON-DEG, LAT+DEG],
    [LON-DEG, LAT-DEG],
]
geojson = json.dumps({
    "type": "FeatureCollection",
    "features": [{"type": "Feature", "properties": {},
                  "geometry": {"type": "Polygon", "coordinates": [coords]}}]
})
params = urllib.parse.urlencode({"dataset": "wpgppop", "year": 2020, "geojson": geojson})
URL = f"https://api.worldpop.org/v1/services/stats?{params}"
print(f"Querying WorldPop: {URL[:100]}...")

Querying WorldPop: https://api.worldpop.org/v1/services/stats?dataset=wpgppop&year=2020&geojson=%7B%22type%22%3A+%22Fea...


## Fetch + Poll (Async Job Handling)

WorldPop may return a `taskid` for async processing. We poll until the job completes.

In [8]:
r = requests.get(URL, timeout=30, headers=HEADERS)
r.raise_for_status()
data = r.json()
print(f"✅ Response: HTTP {r.status_code} | status={data.get('status')}")

# Handle async job
if data.get("status") == "created" and "taskid" in data:
    taskid = data["taskid"]
    print(f"   Job queued (taskid={taskid}). Polling...")
    for attempt in range(12):
        time.sleep(5)
        pr = requests.get(f"https://api.worldpop.org/v1/tasks/{taskid}",
                          timeout=10, headers=HEADERS)
        pr.raise_for_status()
        pdata = pr.json()
        print("-------------")
        print(pdata)
        print("-------------")
        print(f"   Attempt {attempt+1}: status={pdata.get('status')}")
        if pdata.get("status") in ("finished", "OK") and "data" in pdata:
            data = pdata; break
        if pdata.get("status") == "error":
            raise RuntimeError(f"WorldPop job failed: {pdata}")


print(f"\n✅ Job complete")
print(data) 

✅ Response: HTTP 200 | status=created
   Job queued (taskid=05a4dc01-47ba-52fc-892e-223d68ef2819). Polling...
-------------
{'status': 'finished', 'error': True, 'error_message': 'The requested area was too large. Requested 342399.40 km^2 but allowance was 100000.', 'taskid': '05a4dc01-47ba-52fc-892e-223d68ef2819', 'startTime': '2026-04-24 12:52:50.220371', 'endTime': '2026-04-24 12:52:51.822207', 'executionTime': 1}
-------------
   Attempt 1: status=finished
-------------
{'status': 'finished', 'error': True, 'error_message': 'The requested area was too large. Requested 342399.40 km^2 but allowance was 100000.', 'taskid': '05a4dc01-47ba-52fc-892e-223d68ef2819', 'startTime': '2026-04-24 12:52:50.220371', 'endTime': '2026-04-24 12:52:51.822207', 'executionTime': 1}
-------------
   Attempt 2: status=finished
-------------
{'status': 'finished', 'error': True, 'error_message': 'The requested area was too large. Requested 342399.40 km^2 but allowance was 100000.', 'taskid': '05a4dc01-47b

KeyboardInterrupt: 

## Parse + Compute Impact

In [7]:
population = int(data["data"].get("total_population", 0))

depth_factor = round(max(0.05, 1 - DEPTH/500), 2)
felt_radius  = round(10**(0.5*MAG - 0.8), 1)
people_felt  = int(population * 0.40 * depth_factor)

result = {
    "population_in_region":      population,
    "felt_radius_km":            felt_radius,
    "depth_attenuation_factor":  depth_factor,
    "people_likely_felt_shaking": people_felt,
    "high_casualty_risk":        MAG >= 6.5 and DEPTH < 70,
}

print("Population Impact Assessment")
print("-" * 40)
for k, v in result.items():
    print(f"  {k:<35} {v:,}" if isinstance(v, int) else f"  {k:<35} {v}")

KeyError: 'data'

## Visualise

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle(f"WorldPop — Population Impact | M{MAG} depth {DEPTH}km",
             fontsize=12, fontweight='bold')

ax = axes[0]
labels = ["Total\nregion pop.", "Likely felt\nshaking"]
vals   = [population, people_felt]
bars   = ax.bar(labels, vals, color=['#3498db','#e74c3c'], edgecolor='white', width=0.5)
for bar, val in zip(bars, vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(vals)*0.01,
            f"{val:,}", ha='center', fontsize=10)
ax.set_ylabel("People")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f"{int(x):,}"))
ax.set_title("Population exposure", fontweight='bold')

ax = axes[1]; ax.axis('off')
info = [
    ("Bounding box",   f"{DEG*2*111:.0f} × {DEG*2*111:.0f} km"),
    ("Population",     f"{population:,}"),
    ("Felt radius",    f"{felt_radius} km"),
    ("Depth factor",   f"{depth_factor} (1.0=surface)"),
    ("People felt",    f"~{people_felt:,}"),
    ("Casualty risk",  str(result['high_casualty_risk'])),
]
for i, (label, value) in enumerate(info):
    y = 0.9 - i*0.13
    ax.text(0.02, y, f"{label}:", fontweight='bold', fontsize=10, transform=ax.transAxes)
    ax.text(0.45, y, str(value), fontsize=10, transform=ax.transAxes)
ax.set_title("Impact summary", fontweight='bold')

plt.tight_layout()
plt.savefig("nb3_worldpop_result.png", dpi=150, bbox_inches='tight')
plt.show()
print("📊 Saved as nb3_worldpop_result.png")